# 🏗️ Delivery Framework — Hands On: a gated 14-day FDE engagement pipeline

**Problem Type C:** *"Design a delivery framework that takes a customer from scoping doc to
deployed AI agent in under 2 weeks."*

This notebook builds and runs every piece of it: intake refusal, the gate sign-off engine, the
7-stage pipeline, the accelerator library, the metrics, and the negative controls (every way a
gate or stage move is denied). No LLM calls anywhere — every gate decision is deterministic, so every cell below is fast and
exactly reproducible.

## Learning Objectives
In this notebook, you will learn:
1. **Intake refusal** - refuse an engagement with no measurable success metric or no customer SME before the clock starts
2. **Gate sign-off** - see wrong role, missing evidence and an out-of-order gate each denied, and a correct sign-off pass
3. **The 7-stage pipeline** - advance stage by stage, watch a pending gate block a stage move, and see the day-3 escalation fire on its own
4. **The accelerator library** - reuse library assets first and record what had to be custom-built
5. **Metrics and observability** - turn the engagement into numbers (reuse rate, override rate, eval score) and a replayable timeline

## Prerequisites
- `docs/03-src-modules-reference.md` describes each `src/delivery_framework` module (reading only)
- For comparison: `projects/03_enterprise_realtime_capstones/01_enterprise_rag_platform/notebooks/02-hands-on.ipynb` walks through Problem Type B, built the same way (reading only)
- Dependencies installed with `uv sync` in this project's folder
- No keys needed: the notebook makes no LLM or network calls, so this project's `.env` can stay empty

---
## 📦 Step 1: Environment Setup

The setup cell loads this project's `.env` (nothing in it is required), finds the project root, and puts `src/` (the `delivery_framework` package) on `sys.path`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load .env, find the project root, expose src/
# ============================================================================
import sys
import warnings
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# No keys are needed; loading .env keeps the setup identical to the other course notebooks
load_dotenv(find_dotenv(usecwd=True))

# Walk up from the notebook folder until we find src/delivery_framework
ROOT = Path.cwd()
while not (ROOT / "src" / "delivery_framework").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))   # the delivery_framework package

print("project root:", ROOT)
print("✅ Environment ready")

---
## 🚪 Part 1: Intake: refuse before the clock starts

§5.4 names two risk mitigations that both amount to the same thing: **refuse to start** an
engagement that was never going to be measurable. This is the delivery-framework equivalent of the
RAG project's loader refusing a document with no usable ACL — starting a 2-week clock against
something you already know can't be checked is a worse failure than never starting it.

In [ ]:
# ============================================================================
# INTAKE REFUSAL: an unmeasurable request and a request with no SME
# ============================================================================
from delivery_framework.pipeline import intake, ScopingRefused

# An unmeasurable request - refused outright, no Engagement object is even created.
try:
    intake(customer_name="Vague Corp", success_metrics=[],
          data_sources=["zendesk_connector"], customer_sme="u_sme_northwind")
except ScopingRefused as e:
    print("REFUSED:", e)

# No customer SME assigned - also refused (a contractual prerequisite).
try:
    intake(customer_name="No-SME Inc", success_metrics=["first-response time below 5 minutes"],
          data_sources=["zendesk_connector"], customer_sme=None)
except ScopingRefused as e:
    print("REFUSED:", e)

In [ ]:
# ============================================================================
# INTAKE: the Northwind case study is accepted and starts at Scoping, day 1
# ============================================================================
import json
from delivery_framework.config import SETTINGS

case = json.loads((SETTINGS.data_dir / "case_study.json").read_text(encoding="utf-8"))
print(json.dumps(case, indent=2))

engagement = intake(**case)
print(f"\naccepted. stage={engagement.current_stage.label}  day={engagement.day}")

---
## ✅ Part 2: The gate sign-off engine

Every gate decision follows the same three ordered rules, deny-overrides, deliberately the same
shape as `authz.policy.decide()` in the RAG project: wrong role, missing evidence, or an
earlier-stage gate still pending are each an independent hard deny. There is no override path for
any of them in the code.

In [ ]:
# ============================================================================
# PRINCIPALS: everyone who can attempt a sign-off, and their role
# ============================================================================
from delivery_framework import gates
from delivery_framework.identity import get_principal, list_principals

for p in list_principals():
    print(f"{p.user_id:<24}{p.display_name:<42}role={p.role}")

In [ ]:
# ============================================================================
# GATE SIGN-OFF: wrong role and no evidence are denied; the right role with evidence passes
# ============================================================================
fda = get_principal("u_fda_sourav")
sec = get_principal("u_sec_priya")
wrong_hat = get_principal("u_fda_wrong_hat")   # right person, wrong hat - role is still "fda"

# 1. Wrong role - the FDA cannot sign off the security gate, however senior.
d = gates.sign_off(engagement, "security_review_passed", wrong_hat, "trust me")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# 2. Right role, no evidence - an approval with nothing behind it is not a sign-off.
d = gates.sign_off(engagement, "security_review_passed", sec, "")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# 3. Right role, real evidence - passes.
d = gates.sign_off(engagement, "security_review_passed", sec, "SEC-2026-0142, no blocking findings")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

In [ ]:
# ============================================================================
# GATE ORDER: a later gate cannot be signed while an earlier one is pending
# ============================================================================
# Signing a LATER gate while an EARLIER one is still pending - denied, regardless of role/evidence.
sme = get_principal("u_sme_northwind")
d = gates.sign_off(engagement, "golden_set_signed_off", sme, "40 cases approved")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")
print("(golden_set_signed_off needs EVERY earlier gate passed: security_review_passed already is,")
print(" data_access_granted is still pending - all earlier gates are checked, not just the one before it)")

---
## 🛤️ Part 3: The pipeline: a stage cannot be entered while its gate is pending

`advance_stage()` only ever moves to the immediate next stage in `STAGE_ORDER` - there is no code
path that skips one, the same way retrieval in the RAG project has no code path that skips the ACL
filter.

The security gate passed in Part 2, so the first move into Data readiness succeeds. The next move, into Configure, is blocked until `data_access_granted` passes - and because it is still pending on day 3, the escalation check fires on its own.

In [ ]:
# ============================================================================
# STAGE ADVANCE: one allowed move, one blocked move, and the automatic day-3 escalation
# ============================================================================
from delivery_framework import engine

# Allowed: DATA_READINESS's gate (security_review_passed) was signed off in Part 2. Day 3 starts.
d = engine.advance_stage(engagement, to_day=3)
print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}  -> now at '{engagement.current_stage.label}'")

# Blocked: CONFIGURE's gate (data_access_granted) hasn't passed yet.
d = engine.advance_stage(engagement)
print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

engine.check_escalation_triggers(engagement)   # the automatic "escalate day 3" check
for e in engagement.escalations:
    print(f"ESCALATED on day {e.raised_on_day}: {e.reason}")

In [ ]:
# ============================================================================
# DATA READINESS: the SME grants data access, so the move into Configure passes
# ============================================================================
from delivery_framework import accelerators

engine.record_artifact(engagement, "data_access_verification.log", sme.user_id)
d = gates.sign_off(engagement, "data_access_granted", sme,
                   "Read-only Zendesk/Confluence/Salesforce tokens verified live")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

d = engine.advance_stage(engagement, to_day=5)
print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}  -> now at '{engagement.current_stage.label}'")

In [ ]:
# ============================================================================
# ACCELERATORS: reuse library assets first, custom-build only what is missing
# ============================================================================
# Configure, do not code - pull from the accelerator library before building anything custom.
# Connectors for the agreed data sources come from the library too.
accelerators.pull_or_build(engagement, "connector", "zendesk_connector")
accelerators.pull_or_build(engagement, "connector", "confluence_connector")
accelerators.pull_or_build(engagement, "prompt_template", "support_triage_prompt")
accelerators.pull_or_build(engagement, "eval_harness", "golden_set_harness")
accelerators.pull_or_build(engagement, "guardrail_policy", "pii_redaction_policy")
accelerators.pull_or_build(engagement, "guardrail_policy", "northwind_custom_escalation_policy")  # not in the library

for p in engagement.pulls:
    print(f"{p.kind:<20}{p.name:<32}{'reused' if p.reused else 'CUSTOM-BUILT'}")

In [ ]:
# ============================================================================
# GOLDEN SET + EVALUATE: sign the next two gates and move to Shadow mode
# ============================================================================
engine.record_artifact(engagement, "golden_set_v1.json", sme.user_id)
gates.sign_off(engagement, "golden_set_signed_off", sme, "golden_set_v1.json reviewed, 40 cases approved")
d = engine.advance_stage(engagement, to_day=8)
print(f"{'OK' if d.allowed else 'DENY'} -> now at '{engagement.current_stage.label}'")

# Evaluate and iterate.
engine.record_eval_score(engagement, 0.83)
engine.record_artifact(engagement, "eval_baseline_report.pdf", fda.user_id)
gates.sign_off(engagement, "eval_baseline_met", fda, "eval_baseline_report.pdf - 0.83 vs 0.75 agreed baseline")
d = engine.advance_stage(engagement, to_day=10)
print(f"{'OK' if d.allowed else 'DENY'} -> now at '{engagement.current_stage.label}'")

In [ ]:
# ============================================================================
# SHADOW + LIMITED PRODUCTION: human approvals, overrides and the rollback gate
# ============================================================================
# Shadow mode - runs, does not act. Humans compare, some overrides expected early.
for overridden in [True, True, False, False, False]:
    engine.record_human_approval(engagement, overridden=overridden)
engine.record_artifact(engagement, "rollback_runbook.md", fda.user_id)
gates.sign_off(engagement, "rollback_tested", fda, "rollback_runbook.md - tested in staging, <2min")
d = engine.advance_stage(engagement, to_day=12)
print(f"{'OK' if d.allowed else 'DENY'} -> now at '{engagement.current_stage.label}'")

# Limited production - acts, human-approved. Override rate should be falling.
for overridden in [False, False, False, True, False, False]:
    engine.record_human_approval(engagement, overridden=overridden)
engagement.day = 14
print(f"cumulative override rate so far: {engagement.human_approval_overrides}/{engagement.human_approvals_total}")

In [ ]:
# ============================================================================
# GO/NO-GO: the sponsor signs off on the original success metrics, then deploy
# ============================================================================
# Go/no-go - the sponsor signs off only if the ORIGINAL success metrics were actually met.
sponsor = get_principal("u_sponsor_northwind")
engine.record_artifact(engagement, "handover_runbook.md", fda.user_id)
engine.record_artifact(engagement, "dashboards_live.url", fda.user_id)

d = gates.sign_off(engagement, "success_metrics_met", sponsor,
                   "Week-2 metrics: first-response 4m12s, 63% zero-edit sends")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

d = engine.advance_stage(engagement)
print(f"{'OK' if d.allowed else 'DENY'} -> now at '{engagement.current_stage.label}'")

d = engine.mark_deployed(engagement)
print(f"{'OK' if d.allowed else 'DENY'} {d.reason}")

---
## 📊 Part 4: Metrics: turning "the process works" into a number

`metrics.summary()` reads everything the engine recorded on the engagement and reports it as numbers.

In [ ]:
# ============================================================================
# METRICS: the engagement summary
# ============================================================================
from delivery_framework import metrics

for k, v in metrics.summary(engagement).items():
    print(f"{k:<28}{v}")

**Read `accelerator_reuse_rate` carefully:** 5 of 6 pulls in Part 3 came straight from the
library; one guardrail policy was custom-built for Northwind specifically. That 83% is the actual,
numeric answer to §5.1's question — *"productised process, or bespoke heroics?"*

---
## 🔍 Part 5: Observability: the full, replayable timeline

Every gate decision, stage move, escalation, accelerator pull, and artifact - allowed or denied -
is already on `engagement.events`, because every function above logs through
`Engagement.log()` as it runs. This just renders it.

In [ ]:
# ============================================================================
# TIMELINE: render every logged event of the engagement
# ============================================================================
from delivery_framework import observability

print(observability.render_timeline(engagement))

---
## 📝 Summary

In this notebook, we learned:

### 1. Intake refusal
- **Refuse to start rather than start and hope.** An unmeasurable success metric or a missing SME
  is refused at intake, before day 1, the same way an unmappable document is refused at ingest in
  the RAG project.

### 2. Gate sign-off
- **A gate is a decision with a named authority behind it, not a checkbox.** Wrong role is a hard
  deny regardless of seniority - the same instinct as "the LLM is never the enforcement point" in
  the RAG project, applied to people instead of models.

### 3. The 7-stage pipeline
- **There is no code path that skips a stage.** `advance_stage()` only ever moves to the immediate
  next stage in order - the property is structural, not a convention someone has to remember.
- **An automatic check beats a reminder.** The day-3 data-access escalation fires whether anyone
  was watching the tracker or not.
- **A "no-go" is a legitimate outcome.** The framework's job is to make clear *which* gate stopped
  an engagement, not to guarantee every engagement deploys in 14 days.

### 4. The accelerator library
- **"Productised vs. bespoke" is a ratio, not a claim.** `accelerator_reuse_rate` is the number
  that actually answers the question the prep doc poses in §5.1.

### 5. Metrics and observability
- Every gate decision, stage move, escalation, pull and artifact - allowed or denied - is on
  `engagement.events`, so the timeline is a replayable record, not a report written afterwards.

### Next Steps
This is the project's only notebook.
- **`scripts/run_engagement_demo.py`** and **`scripts/demo_gate_failure.py`**: the same happy path and negative controls as scripts (`uv run python scripts/<name>.py` from the project folder).
- **`README.md`**: how the framework maps to the system-design question, and its known gaps.